<h3>Install Kaggle and torchmetrics</h3>

In [ ]:
pip install kagglehub

In [ ]:
pip install torchmetrics

<h3>Load dataset</h3>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("paultimothymooney/chest-xray-pneumonia")

print("Path to dataset files:", path)

<h3>Move your dataset</h3>

In [ ]:
mv {path} {your-workspace}

<h3>Device</h3>

In [ ]:
import torch
device_name = "cuda"
if torch.mps.is_available():
    device_name = "mps"
elif torch.cuda.is_available():
    device_name = "cuda"
else:
    device_name = "cpu"
print(device_name)


<h3>Define dataset</h3>

In [ ]:
import torch

from torchvision.io import read_image, decode_image, ImageReadMode
import torchvision.transforms as transforms
import os
class PneumoniaDataset(torch.utils.data.Dataset):


    
    def __init__(self, dataset_dir_path, augment=True, image_size=(128,128)):
        super().__init__()
        self.images_paths = []
        self.classes = []
        self.cache = []
        self.image_size = image_size

        self.augment = augment


        class_counter = 0
        self.resize = transforms.Resize(image_size)
        #self.normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
        for class_dir in os.listdir(dataset_dir_path):
            class_dir_path = os.path.join(dataset_dir_path,class_dir)
            if os.path.isdir(class_dir_path):
                for filename in os.listdir(class_dir_path):
                    
                    self.images_paths.append(os.path.join(class_dir_path, filename))
                    
                    self.classes.append(class_counter)
                    
                class_counter+=1
        
        self.random_transform = transforms.Compose(
        [
            
            transforms.RandomHorizontalFlip(),
            transforms.RandomAffine(
                degrees=30,
                translate=(0.1, 0.1),
                scale=None,
                shear=10,
                fill=0
            ),
        
        
            
            

        ])
        
    def __getitem__(self, index):
        image = None
        
        try:
            image = self.cache[index]
            

        except IndexError:
            image = read_image(self.images_paths[index], ImageReadMode.GRAY).float()/255
            image = self.resize(image)
            
            self.cache.append(image)
            
        if self.augment:
            image = self.random_transform(image)
        return (image, self.classes[index])
    
    def __len__(self):
        return len(self.images_paths)



<h3>Define model</h3>

In [ ]:
import torch
from torch.nn import Linear, Conv2d, ReLU, Sigmoid,MaxPool2d, AvgPool2d, Flatten, BatchNorm2d, Dropout

from torchmetrics.classification import BinaryAccuracy
from torch.optim.lr_scheduler import ReduceLROnPlateau

class PneumoniaDetectionModel(torch.nn.Module):

    def __init__(self, lr=0.001):
        super().__init__()
        self.loss_fn = torch.nn.BCEWithLogitsLoss()
        
        self.conv_model = torch.nn.Sequential(
            Conv2d(1,64, 3),
            BatchNorm2d(64),
            ReLU(),
            Conv2d(64,64, 3),
            BatchNorm2d(64),
            ReLU(),
            MaxPool2d((2,2)),

            Conv2d(64,128, 3),
            BatchNorm2d(128),
            ReLU(),
            Conv2d(128,128, 3),
            BatchNorm2d(128),
            ReLU(),
            MaxPool2d((2,2)),

            Conv2d(128,256, 3),
            BatchNorm2d(256),
            ReLU(),
            Conv2d(256,256, 3),
            BatchNorm2d(256),
            ReLU(),
            MaxPool2d((2,2)),


            Conv2d(256,512, 3),
            BatchNorm2d(512),
            ReLU(),
            Conv2d(512,512, 3),
            BatchNorm2d(512),
            ReLU(),
            MaxPool2d((2,2)),
            

            Conv2d(512,512, 3, stride=2),
            BatchNorm2d(512),
            ReLU(),

            
            
        )
        
        self.classifier = torch.nn.Sequential(
            Flatten(),
            
            Linear(512, 1),
        
        )

        
        self.optimizer = torch.optim.Adam(self.parameters(),lr=lr)
        self.accuracy = BinaryAccuracy()
        self.to(device_name)
        self.scheduler = ReduceLROnPlateau(
            self.optimizer,
            mode='min',      # 'min' for loss, 'max' for accuracy
            factor=0.5,      # multiply LR by this factor
            patience=3,      # wait N epochs with no improvement before reducing LR
           
        )
    def forward(self, images):
        features = self.conv_model(images)
        #print(features.size())
        output = self.classifier(features)
        return output
    

    def evaluate(self, data_loader):
        self.eval()
        self.accuracy.reset()
        val_loss = 0
        with torch.no_grad():
            for batch in iter(data_loader):
                
                x = batch[0].to(device_name)
                y = batch[1].to(device_name).float()

                output = self.forward(x)
                output = output.view(output.size(0), 1)
                
                loss = self.loss_fn(output.squeeze(), y.squeeze())
                self.accuracy(output.squeeze(), y.squeeze())
                val_loss += loss.item()
                
                
        self.train(True)
        return (val_loss/len(data_loader), self.accuracy.compute())
        
    def fit(self, train_loader, val_loader, epochs=10):
        best_loss = 10
        for i in range(0,epochs):
            epoch_loss = 0
            self.accuracy.reset()
            
        
            for batch in iter(train_loader):
                
                x = batch[0].to(device_name)
                y = batch[1].to(device_name).float()

                output = self.forward(x)
                output = output.view(output.size(0), 1)
                
                loss = self.loss_fn(output.squeeze(), y.squeeze())
                
                epoch_loss += loss.item()
                
                self.accuracy(output.squeeze(), y.squeeze())
                
                self.optimizer.zero_grad()

                loss.backward()
                self.optimizer.step()
            
            epoch_acc = self.accuracy.compute()
            val_loss, val_acc = self.evaluate(val_loader)

            
            if best_loss > val_loss:
                best_loss = val_loss
                torch.save(self.state_dict(), f"model-loss-{val_loss:4f}-acc-{val_acc:4f}")

            self.scheduler.step(val_loss)

            print(f"Epoch {i+1} Loss:{epoch_loss/len(train_loader):.4f} Accuracy:{epoch_acc:.4f}, Val Loss:{val_loss:.4f} Val Accuracy:{val_acc:.4f} LR {self.optimizer.param_groups[0]['lr']}")
      

<h3>Create datasets</h3>

In [ ]:
import torch
from torch.utils.data import DataLoader


TRAIN_DATA_DIR = "./chest_xray/train"
TEST_DATA_DIR = "./chest_xray/test"
VAL_DATA_DIR = "./chest_xray/val"

IMG_SIZE = (128,128)

train_dataset = PneumoniaDataset(TRAIN_DATA_DIR, image_size=IMG_SIZE)
test_dataset = PneumoniaDataset(TEST_DATA_DIR,image_size=IMG_SIZE, augment=False)
val_dataset = PneumoniaDataset(VAL_DATA_DIR,image_size=IMG_SIZE, augment=False)



train_dataloader = DataLoader(train_dataset, batch_size=16,num_workers=10, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=16, num_workers=10)
test_dataloader = DataLoader(test_dataset, batch_size=16)



<h3>Create model</h3>

In [ ]:
model = PneumoniaDetectionModel(lr=0.0009)

<h3>Train model</h3>

In [ ]:
model.fit(train_dataloader, val_dataloader, epochs=15)

<h3>Load the best model</h3>

In [ ]:
model.load_state_dict(torch.load("best-model"))

<h3>Evaluate on the test set</h3>

In [ ]:
print(model.evaluate(test_dataloader))